In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer

from src.data.dataset import PreferenceDataset, PreferenceCollator, create_dataloader


MODEL_NAME = "gpt2"
MAX_LENGTH = 512
SAMPLE_SIZE = 20


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# Load only 20 examples
if SAMPLE_SIZE is not None:
    raw_dataset = load_dataset(
        "Anthropic/hh-rlhf",
        split=f"train[:{SAMPLE_SIZE}]",
    )
else:
    raw_dataset = load_dataset(
        "Anthropic/hh-rlhf",
        split="train",
    )


dataset = PreferenceDataset(
    data=raw_dataset,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)


# ------------------------------------------------------------
# Take a few individual examples
# ------------------------------------------------------------

examples = [
    dataset[0],
    dataset[1],
    dataset[2],
]


# ------------------------------------------------------------
# Create collator
# ------------------------------------------------------------

collator = PreferenceCollator(
    pad_token_id=tokenizer.pad_token_id
)


# ------------------------------------------------------------
# Create a batch manually
# ------------------------------------------------------------

batch = collator(examples)


# ------------------------------------------------------------
# Inspect batch
# ------------------------------------------------------------

print("\nChosen input IDs:")
print(batch["chosen_input_ids"].shape)

print("\nChosen attention mask:")
print(batch["chosen_attention_mask"].shape)

print("\nRejected input IDs:")
print(batch["rejected_input_ids"].shape)

print("\nRejected attention mask:")
print(batch["rejected_attention_mask"].shape)


# ------------------------------------------------------------
# Inspect actual attention masks
# ------------------------------------------------------------

print("\nChosen attention mask:")
print(batch["chosen_attention_mask"])

print("\nRejected attention mask:")
print(batch["rejected_attention_mask"])


Chosen input IDs:
torch.Size([3, 202])

Chosen attention mask:
torch.Size([3, 202])

Rejected input IDs:
torch.Size([3, 196])

Rejected attention mask:
torch.Size([3, 196])

Chosen attention mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 

In [2]:
dataloader = create_dataloader(
    dataset=dataset,
    tokenizer=tokenizer,
    batch_size=4,
    shuffle=False,
    num_workers=0,
)

In [3]:
batch = next(iter(dataloader))

In [4]:
print("\nBatch shapes:")

print(
    "Chosen input IDs:",
    batch["chosen_input_ids"].shape,
)

print(
    "Chosen attention mask:",
    batch["chosen_attention_mask"].shape,
)

print(
    "Rejected input IDs:",
    batch["rejected_input_ids"].shape,
)

print(
    "Rejected attention mask:",
    batch["rejected_attention_mask"].shape,
)


Batch shapes:
Chosen input IDs: torch.Size([4, 202])
Chosen attention mask: torch.Size([4, 202])
Rejected input IDs: torch.Size([4, 196])
Rejected attention mask: torch.Size([4, 196])


In [5]:
print("\nChosen sequence lengths from attention mask:")

print(
    batch["chosen_attention_mask"].sum(dim=1)
)

print("\nRejected sequence lengths from attention mask:")

print(
    batch["rejected_attention_mask"].sum(dim=1)
)


Chosen sequence lengths from attention mask:
tensor([202, 107,  53, 101])

Rejected sequence lengths from attention mask:
tensor([196, 117, 181, 106])


## Reward Model testing

In [6]:
import torch
from transformers import AutoTokenizer

from src.data.dataset import PreferenceDataset
from src.data.dataset import create_dataloader
from src.models.reward_model import GPT2RewardModel

from datasets import load_dataset


MODEL_NAME = "gpt2"
MAX_LENGTH = 512
SAMPLE_SIZE = 20
BATCH_SIZE = 4


# ============================================================
# Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ============================================================
# Load small dataset
# ============================================================

raw_dataset = load_dataset(
    "Anthropic/hh-rlhf",
    split=f"train[:{SAMPLE_SIZE}]",
)


dataset = PreferenceDataset(
    data=raw_dataset,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)


# ============================================================
# DataLoader
# ============================================================

dataloader = create_dataloader(
    dataset=dataset,
    tokenizer=tokenizer,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)


batch = next(iter(dataloader))


# ============================================================
# Model
# ============================================================

model = GPT2RewardModel(
    model_name=MODEL_NAME
)


# ============================================================
# Forward pass
# ============================================================

with torch.no_grad():

    chosen_rewards = model(
        input_ids=batch["chosen_input_ids"],
        attention_mask=batch["chosen_attention_mask"],
    )

    rejected_rewards = model(
        input_ids=batch["rejected_input_ids"],
        attention_mask=batch["rejected_attention_mask"],
    )


# ============================================================
# Inspect
# ============================================================

print("\nChosen rewards:")
print(chosen_rewards)

print("\nRejected rewards:")
print(rejected_rewards)

print("\nShapes:")

print(
    "chosen_rewards:",
    chosen_rewards.shape,
)

print(
    "rejected_rewards:",
    rejected_rewards.shape,
)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

c:\Projects\testenv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\manin\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)



Chosen rewards:
tensor([-2.3810, -2.2455, -2.7978, -2.1226])

Rejected rewards:
tensor([-2.2680, -1.9963, -2.3768, -2.0792])

Shapes:
chosen_rewards: torch.Size([4])
rejected_rewards: torch.Size([4])
